<a href="https://colab.research.google.com/github/LavikLee/LibreTV-32298/blob/main/%E6%8F%90%E5%8F%96%E4%B8%80%E8%87%B4%E6%80%A7%E9%A2%84%E6%9C%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install akshare

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 31.9 MB/s eta 0:00:00
  Created wheel for jsonpath: filename=jsonpath-0.82.2-py3-none-any.whl size=5615 sha256=c01d5052fc2b4f658fc23561effdbf379665e9ea17c521f70e5d94c4f3293693
  Stored in directory: /root/.cache/pip/wheels/73/76/e2/980a29341fe37a583ada29594ed529708d5e8e2c0f9d97c3cc
Successfully built jsonpath


# From Gemini

In [ ]:
import akshare as ak
import pandas as pd
import time
import random
import sys
import os
import shutil

# 尝试导入 Colab 特有模块
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_stock_forecast_ths():
    """
    使用 ak.stock_profit_forecast_ths 获取数据。
    优化：修复进度条重叠、支持断点续传、支持测试模式限制抓取量。
    """
    target_years = ["2026", "2027", "2028"]
    drive_path = "/content/drive/MyDrive/stock_data/"
    checkpoint_file = "processed_list.txt"
    local_files = {year: f"stock_forecast_inst_{year}.txt" for year in target_years}

    # --- 配置项：测试限制 ---
    # 如果只想测试前 100 只，请设置为 100；如果想全量运行，请设置为 None
    test_limit = 100

    # 1. 挂载 Google Drive
    if IN_COLAB:
        if not os.path.exists('/content/drive'):
            print("正在挂载 Google Drive...")
            drive.mount('/content/drive')
        if not os.path.exists(drive_path):
            os.makedirs(drive_path)

    # 2. 恢复进度
    processed_codes = set()
    if IN_COLAB:
        for f_name in list(local_files.values()) + [checkpoint_file]:
            cloud_f = os.path.join(drive_path, f_name)
            if os.path.exists(cloud_f):
                shutil.copy(cloud_f, f_name)

        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                processed_codes = set(line.strip() for line in f if line.strip())
            print(f"成功恢复：已处理 {len(processed_codes)} 只股票。")

    # 3. 获取全量代码
    print("正在获取最新 A 股代码列表...")
    try:
        stock_list_df = ak.stock_info_a_code_name()
        all_codes = [str(c).zfill(6) for c in stock_list_df['code'].tolist()]
    except Exception as e:
        print(f"获取列表失败: {e}")
        return

    # 过滤掉已处理的代码
    pending_codes = [c for c in all_codes if c not in processed_codes]

    # --- 应用测试限制 ---
    if test_limit is not None:
        pending_codes = pending_codes[:test_limit]

    total_run = len(pending_codes)

    if total_run == 0:
        print("所有任务已完成或在本次限制范围内无新任务。")
        return

    # 初始化本地文件
    for f_name in list(local_files.values()) + [checkpoint_file]:
        if not os.path.exists(f_name):
            with open(f_name, 'w', encoding='utf-8') as f: pass

    success_count = 0
    start_time = time.time()
    mode_str = f"前 {test_limit} 只" if test_limit else "全量"
    print(f"开始执行 ({mode_str})：待处理 {total_run} 只，目标：{', '.join(target_years)}")
    print("-" * 70)

    for index, code in enumerate(pending_codes):
        current_idx = index + 1
        market_type = "1" if code.startswith(('6', '68')) else "0"

        # 计算时间
        elapsed = time.time() - start_time
        avg = elapsed / current_idx
        rem = avg * (total_run - current_idx)

        # 刷新进度条
        status_msg = f"进度: [{current_idx}/{total_run}] {code} | 累计成功: {success_count} | 剩余时间: {rem:.1f}s"
        sys.stdout.write(f"\r{status_msg}\033[K")
        sys.stdout.flush()

        # 定期同步与冷却
        if index > 0 and index % 40 == 0:
            if IN_COLAB:
                for f_name in list(local_files.values()) + [checkpoint_file]:
                    shutil.copy(f_name, os.path.join(drive_path, f_name))

            for i in range(10, 0, -1):
                sys.stdout.write(f"\r[系统] 正在同步数据并冷却... 剩余 {i} 秒\033[K")
                sys.stdout.flush()
                time.sleep(1)
            sys.stdout.write(f"\r{status_msg}\033[K")
            sys.stdout.flush()

        time.sleep(random.uniform(0.6, 1.2))

        try:
            df = ak.stock_profit_forecast_ths(symbol=code, indicator="预测年报净利润")
            if df is not None and not df.empty:
                df.columns = [c.strip() for c in df.columns]
                any_saved = False
                for year in target_years:
                    row = df[df['年度'].astype(str).str.contains(year)]
                    if not row.empty:
                        val = row.iloc[0]['均值']
                        inst = str(row.iloc[0]['预测机构数']).strip()
                        if not pd.isna(val) and str(val).strip() != '--':
                            line = f"{market_type}|{code}|{inst}|{float(val):.2f}\n"
                            with open(local_files[year], 'a', encoding='utf-8') as f:
                                f.write(line)
                                f.flush()
                            any_saved = True
                if any_saved: success_count += 1

            # 记录进度
            with open(checkpoint_file, 'a', encoding='utf-8') as f:
                f.write(f"{code}\n")
                f.flush()

        except Exception:
            time.sleep(2)
            continue

    # 最终同步
    if IN_COLAB:
        print("\n" + "-" * 70)
        print("执行最终备份...")
        for f_name in list(local_files.values()) + [checkpoint_file]:
            shutil.copy(f_name, os.path.join(drive_path, f_name))
        print(f"任务完成！已成功同步至 Google Drive。")

if __name__ == "__main__":
    get_stock_forecast_ths()

正在挂载 Google Drive...
Mounted at /content/drive
正在获取最新 A 股代码列表...
获取列表失败: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


# From ChatGPT

In [ ]:
import akshare as ak
import pandas as pd
import time
import random
import sys
import os
import shutil

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def repair_data_files(local_files, checkpoint_set):
    """
    用 checkpoint 修复数据文件：
    删除不在 checkpoint 中的记录
    """
    print("正在进行数据一致性修复...")

    for year, file_path in local_files.items():
        if not os.path.exists(file_path):
            continue

        new_lines = []
        removed = 0

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) < 2:
                    continue

                code = parts[1]

                if code in checkpoint_set:
                    new_lines.append(line)
                else:
                    removed += 1

        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(new_lines)

        print(f"{file_path} 修复完成，移除 {removed} 条异常数据")


def get_stock_forecast_ths():
    target_years = ["2026", "2027", "2028"]
    drive_path = "/content/drive/MyDrive/stock_data/"
    checkpoint_file = "processed_list.txt"
    local_files = {year: f"stock_forecast_inst_{year}.txt" for year in target_years}

    test_limit = None
    BATCH_SIZE = 20   # ⭐ 每100条同步一次

    # ========================
    # Drive 初始化
    # ========================
    if IN_COLAB:
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        os.makedirs(drive_path, exist_ok=True)

    # ========================
    # 拉取云端文件
    # ========================
    if IN_COLAB:
        for f in list(local_files.values()) + [checkpoint_file]:
            cloud_f = os.path.join(drive_path, f)
            if os.path.exists(cloud_f):
                shutil.copy(cloud_f, f)

    # ========================
    # 读取 checkpoint
    # ========================
    processed_codes = set()

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            processed_codes = set(line.strip() for line in f if line.strip())

    print(f"已恢复 {len(processed_codes)} 条记录")

    # ========================
    # ⭐ 数据修复（关键）
    # ========================
    repair_data_files(local_files, processed_codes)

    # ========================
    # 获取股票列表
    # ========================
    stock_list_df = ak.stock_info_a_code_name()
    all_codes = [str(c).zfill(6) for c in stock_list_df['code'].tolist()]

    pending_codes = [c for c in all_codes if c not in processed_codes]

    if test_limit:
        pending_codes = pending_codes[:test_limit]

    total = len(pending_codes)
    if total == 0:
        print("无任务")
        return

    # ========================
    # buffer
    # ========================
    buffers = {year: [] for year in target_years}
    checkpoint_buffer = []

    success_count = 0
    start_time = time.time()

    print(f"开始处理 {total} 条")

    for idx, code in enumerate(pending_codes, 1):

        market_type = "1" if code.startswith(('6', '68')) else "0"

        elapsed = time.time() - start_time
        speed = idx / elapsed if elapsed > 0 else 0
        eta = (total - idx) / speed if speed > 0 else 0

        sys.stdout.write(
            f"\r[{idx}/{total}] {code} | 成功:{success_count} | ETA:{eta:.1f}s"
        )
        sys.stdout.flush()

        time.sleep(random.uniform(0.6, 1.2))

        try:
            df = ak.stock_profit_forecast_ths(
                symbol=code, indicator="预测年报净利润"
            )

            if df is None or df.empty:
                checkpoint_buffer.append(code)
                continue

            df.columns = [c.strip() for c in df.columns]
            df['年度'] = df['年度'].astype(str)

            any_saved = False

            for year in target_years:
                row = df[df['年度'] == year]

                if not row.empty:
                    val = row.iloc[0]['均值']
                    inst = str(row.iloc[0]['预测机构数']).strip()

                    if pd.notna(val) and str(val).strip() != '--':
                        buffers[year].append(
                            f"{market_type}|{code}|{inst}|{float(val):.2f}\n"
                        )
                        any_saved = True

            if any_saved:
                success_count += 1

            checkpoint_buffer.append(code)

        except Exception as e:
            print(f"\n[ERROR] {code}: {e}")
            time.sleep(2)

        # ========================
        # ⭐ 每100条提交一次
        # ========================
        if idx % BATCH_SIZE == 0:

            # 写数据文件
            for year, lines in buffers.items():
                if lines:
                    with open(local_files[year], 'a', encoding='utf-8') as f:
                        f.writelines(lines)
                    buffers[year].clear()

            # 写 checkpoint
            if checkpoint_buffer:
                with open(checkpoint_file, 'a', encoding='utf-8') as f:
                    f.write("\n".join(checkpoint_buffer) + "\n")
                checkpoint_buffer.clear()

            # 同步 Drive
            if IN_COLAB:
                print("\n[同步] 正在上传到 Drive...")
                for f in list(local_files.values()) + [checkpoint_file]:
                    shutil.copy(f, os.path.join(drive_path, f))

    # ========================
    # 收尾写入
    # ========================
    for year, lines in buffers.items():
        if lines:
            with open(local_files[year], 'a', encoding='utf-8') as f:
                f.writelines(lines)

    if checkpoint_buffer:
        with open(checkpoint_file, 'a', encoding='utf-8') as f:
            f.write("\n".join(checkpoint_buffer) + "\n")

    # 最终同步
    if IN_COLAB:
        print("\n最终同步到 Drive...")
        for f in list(local_files.values()) + [checkpoint_file]:
            shutil.copy(f, os.path.join(drive_path, f))

    print("\n完成")


if __name__ == "__main__":
    get_stock_forecast_ths()

已恢复 0 条记录
正在进行数据一致性修复...
开始处理 5528 条
[14/5528] 000019 | 成功:3 | ETA:11922.7s